# BayMorph — multi-galaxy analysis

Run the same BayMorph configuration template on a directory of FITS images. For each object, only the object name and image path are changed.

## Setup

In [ ]:
from pathlib import Path
import copy
import os

import pandas as pd

from baymorph import BayMorphRunner, load_resolved_config

if Path.cwd().name == "notebooks":
    os.chdir("..")

## User settings

In [ ]:
BASE_CONFIG = Path("configs/real_galaxy.yaml")
DATA_DIR = Path("data")
PATTERN = "*.fits"
BAND = None
MAX_OBJECTS = None

## Discover input images

In [ ]:
fits_files = sorted(DATA_DIR.glob(PATTERN))

if MAX_OBJECTS is not None:
    fits_files = fits_files[:MAX_OBJECTS]

print(f"Found {len(fits_files)} FITS files.")
fits_files

## Run BayMorph

In [ ]:
base_config = load_resolved_config(BASE_CONFIG)
results = []

for image_path in fits_files:
    config = copy.deepcopy(base_config)
    object_name = image_path.stem

    config["object"]["name"] = object_name
    config["data"]["image_path"] = str(image_path)

    separator = "=" * 72

    print(
        f"\n{separator}\n"
        f"{object_name}\n"
        f"{image_path}\n"
        f"{separator}"
    )

    runner = BayMorphRunner(config)
    runner.prepare_image()

    if config["psf"]["type"] == "empirical":
        runner.select_psf_candidates()
        runner.build_psf()

    fit_products = runner.fit(return_result=True)
    runner.diagnostics()

    products = (
        fit_products.items()
        if isinstance(fit_products, dict)
        else [(fit_products.model_name, fit_products)]
    )

    for model_name, product in products:
        results.append({
            "object": object_name,
            "band": BAND,
            "model": model_name,
            "chi2": product.chi2,
            "reduced_chi2": product.reduced_chi2,
            "ndof": product.ndof,
        })

## Summary

In [ ]:
summary = pd.DataFrame(results)
summary

In [ ]:
output_path = Path(base_config["outputs"]["root"]) / "multi_galaxy_summary.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(output_path, index=False)
print("Saved:", output_path)